# Inroduction

In this Notebook, we will manipulate some doubly-shifted textures and experiment with peak detection, minimal patch search, and homography estimation based on local affinities.

1. The first part covers affine deformations and their rectification based on a single affinity.

2. The second part covers homographic deformations, the calculation of local affinities, and homography estimation.

3. The third part contains a study that shows the error of this estimation over a range of viewing angles and calculates the pixel error between the actual homography and the estimated homography.

We invite you to manipulate the calculation and distortion parameters for a dynamic and satisfying experience in order to fully understand their impact on different textures.

In [ ]:
import numpy as np
from PIL import Image
import cv2 as cv
from IPython.display import display
import importlib
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import operations as oprt
import autocorrelation
import graph_Ai_Jc_energy
import ghostseal_generator
import deformation_generator
import find_min_stable_patch_size
import rectification_energy
import rectification 

importlib.reload(oprt)
importlib.reload(autocorrelation)
importlib.reload(graph_Ai_Jc_energy)
importlib.reload(ghostseal_generator)
importlib.reload(deformation_generator)
importlib.reload(find_min_stable_patch_size)
importlib.reload(rectification_energy)
importlib.reload(rectification)


In [ ]:
from autocorrelation import autocorrelation_display, correlation_nomalisee
from ghostseal_generator import generate_white_noise_and_shifts, gen_random_binary_texture
from deformation_generator import general_deformer
from find_min_stable_patch_size import find_min_stable_patch_size_centered
from rectification_energy import local_affinity_via_min_patch_at,choose_best_affinity_patch
import detect_shifts as shm
from rectification import rectification
from optimization_better import optimize

In [ ]:
def display_img(image, width=300):
    low, high = np.percentile(image, (1, 99))  
    img_clip = np.clip(image, low, high)
    img_clip = 255 * (img_clip - low) / (high - low)
    img_clip = img_clip.astype(np.uint8)

    pil_img = Image.fromarray(img_clip)
    display(pil_img.resize(
        (width, int(pil_img.height * width / pil_img.width))
    ))


def frobenius_norm(A, H_inv):
    A_lin_persp = A[:, :2]
    H_lin_persp = H_inv[:, :2]
    return np.linalg.norm(A_lin_persp - H_lin_persp, "fro")


def crop_and_align(image_ref, copies):
    ref_resized, crop_resized = oprt.padding_zone_corr(image_ref,copies)
    corr = correlation_nomalisee(ref_resized, crop_resized)

    h_, w_ = corr.shape
    h, w = copies.shape
    x, y = shm.find_peak_position(corr, method='centroid')
    dx = w / 2 - x - (w - w_) / 2
    dy = h / 2 - y - (h - h_) / 2
    H_affine = np.eye(3)
    H_affine[0, 2] -= dx
    H_affine[1, 2] -= dy

    transformed_crop = oprt.apply_homography(crop_resized, H_affine, (h_, w_),0)
    return transformed_crop , corr,ref_resized, crop_resized


In [ ]:
def jacobian_homography(H, x, y, eps=1e-12):
    h11,h12,h13 = H[0,0], H[0,1], H[0,2]
    h21,h22,h23 = H[1,0], H[1,1], H[1,2]
    h31,h32,h33 = H[2,0], H[2,1], H[2,2]

    w = h31*x + h32*y + h33
    w2 = (w*w) + eps

    u = h11*x + h12*y + h13
    v = h21*x + h22*y + h23

    dxp_dx = (h11*w - u*h31) / w2
    dxp_dy = (h12*w - u*h32) / w2
    dyp_dx = (h21*w - v*h31) / w2
    dyp_dy = (h22*w - v*h32) / w2

    return dxp_dx, dxp_dy, dyp_dx, dyp_dy, w

def cond_number_map_homography(H, out_shape, use_inverse=True, eps=1e-12):
    H = np.asarray(H, dtype=float)
    if use_inverse:
        H = np.linalg.inv(H)

    Hh, Hw = out_shape
    y, x = np.mgrid[0:Hh, 0:Hw]
    x = x.astype(float)
    y = y.astype(float)

    dxp_dx, dxp_dy, dyp_dx, dyp_dy, w = jacobian_homography(H, x, y, eps=eps)

    J = np.stack([
        np.stack([dxp_dx, dxp_dy], axis=-1),
        np.stack([dyp_dx, dyp_dy], axis=-1)
    ], axis=-2)  

    s = np.linalg.svd(J, compute_uv=False)
    s0 = s[..., 0]
    s1 = s[..., 1]

    cond = s0 / (s1 + eps)
    mask = np.abs(w) < 1e-6
    cond = np.where(mask, np.nan, cond)

    return cond

def plot_cond_heatmap(cond_map, img_warped=None, title="Conditionnement of Jacobians (homography)",
                      vmin=None, vmax=None, log_scale=True, alpha=0.55):
    data = cond_map.copy()

    plt.figure(figsize=(10, 6))

    if img_warped is not None:
        plt.imshow(img_warped, cmap="gray")
        plt.imshow(data, cmap="viridis", alpha=alpha, vmin=vmin, vmax=vmax)
    else:
        plt.imshow(data, cmap="viridis", vmin=vmin, vmax=vmax)

    plt.colorbar(label=("cond(J)" if log_scale else "cond(J)"))
    plt.title(title)
    plt.axis("off")
    plt.tight_layout()
    plt.show()

# 1. Affinity Case

## 1.1. generate and deforme textures

### 1.1.1. white noise doubly-shifted texture

In this experiment, we generate a synthetic texture with a controlled periodic structure by superimposing spatially shifted versions of white noise.

In [ ]:
shape = 1000
u_shift = (60, 0)
v_shift = (0, 60)

seed=21353456

noise_texture = generate_white_noise_and_shifts(
        shape, shape, u_shift, v_shift,seed=seed
    )
noise_texture = cv.GaussianBlur(noise_texture,(11,11),0)
display_img(noise_texture)



### 1.1.2. random binary dots doubly-shifted texture

We generate synthetic textures designed to produce **binary dot patterns with controlled spatial correlations**.
The method is based on the superposition of shifted random binary patterns, followed by morphological processing.

In [ ]:
shape = 1000
dilation_size_i = 1
density = 0.01
angle_shift = 90
norm_shift = 60

binary_dots_texture = gen_random_binary_texture(shape,shape, dilation_size_i,density, angle_shift, norm_shift)
display_img(binary_dots_texture)

## 1.2. texture deformation

### 1.2 Texture deformation

Here, the affinity deformation matrix *A* given by:
$$
A =
\begin{bmatrix}
\lambda & 0 \\
0 & \lambda
\end{bmatrix}
R_{-\theta}
\begin{bmatrix}
\tilde{t} & 0 \\
0 & 1
\end{bmatrix}
R_{\theta} R_{\alpha}.
$$


**Where:**

- **$\lambda$** is an isotropic scale factor  
- **$\tilde{t}$** is the tilt factor  
- **$\theta$** is the orientation angle of the tilt  
- **$\R_{\alpha}$** denotes a planar rotation matrix:
$$
R_{\alpha}
=
\begin{bmatrix}
\cos \alpha & -\sin \alpha \\
\sin \alpha & \cos \alpha
\end{bmatrix}.
$$

we apply this deformation to both textures

In [ ]:
scale = 1
rotation = 10
tilt = 0.6
tilt_orientation = 30
inclination_x = 0 #parameter to be seen later in the homography section
inclination_y = 0 #parameter to be seen later in the homography section

In [ ]:
noise_deformed, noise_matrix = general_deformer(noise_texture,scale, rotation, tilt, tilt_orientation, inclination_x, inclination_y)
display_img(noise_deformed)


In [ ]:
binary_deformed, binary_matrix = general_deformer(binary_dots_texture,scale, rotation, tilt, tilt_orientation, inclination_x, inclination_y)
display_img(binary_deformed)

## 1.3. autocorrelation

When calculating the autocorrelation of the two doubly-shifted textures, we notice the appearance of seven peaks showing the intercorrelation of the superimposed copies, as demonstrated below.

$$
\mathcal{R}_{J}(\boldsymbol{\tau})
=
\sum_{\boldsymbol{x}\in P}
\Big(
3\,\mathcal{R}_{T}(\boldsymbol{\tau})
+ \mathcal{R}_{T}(\boldsymbol{\tau}+a)
+ \mathcal{R}_{T}(\boldsymbol{\tau}+b)
+ \mathcal{R}_{T}(\boldsymbol{\tau}-a)
+ \mathcal{R}_{T}(\boldsymbol{\tau}-b)
+ \mathcal{R}_{T}(\boldsymbol{\tau}-(a-b))
+ \mathcal{R}_{T}(\boldsymbol{\tau}+(a-b))
\Big).
$$

Since $ \mathcal{R}_{T} $ reaches its maximum at $ \boldsymbol{\tau} = 0 $,
the function $ \mathcal{R}_{J} $ exhibits a central peak at $ \boldsymbol{\tau} = 0 $
and *six* satellite peaks located at
$$
\pm a,\quad \pm b,\quad \pm (a-b).
$$

These extrema correspond to the vertices of an affine hexagon, with $ a \neq b $.


In [ ]:
noise_autocorrelation = autocorrelation_display(noise_texture)
deformed_noise_autocorrelation = autocorrelation_display(noise_deformed)
display_img(noise_autocorrelation)
display_img(deformed_noise_autocorrelation)

In [ ]:
binary_autocorrelation = autocorrelation_display(binary_dots_texture)
binary_deformed_autocorrelation = autocorrelation_display(binary_deformed)
display_img(binary_autocorrelation)
display_img(binary_deformed_autocorrelation)

## 1.4. Peaks detection and find min stable patch

Peak detection in the autocorrelation image is performed by searching
for a number $k$ of local maxima in the image. To achieve sub-pixel accuracy,
the two peaks $\mathbf{u}_{\text{int}}$ and $\mathbf{v}_{\text{int}}$
are jointly refined in order to obtain sub-pixel precision.
We seek the displacements $(\mathbf{r}_1, \mathbf{r}_2) \in [-h,h]^2$
that maximize:

$$
(\hat{\mathbf{r}}_1, \hat{\mathbf{r}}_2)
= \arg\max_{\mathbf{r}_1, \mathbf{r}_2 \in [-h,h]^2}
\Big[
  \mathcal{R}(\mathbf{u}_{\text{int}} + \mathbf{r}_1)
  + \mathcal{R}(\mathbf{v}_{\text{int}} + \mathbf{r}_2)
  + \mathcal{R}\big(
     (\mathbf{u}_{\text{int}} + \mathbf{r}_1)
     - (\mathbf{v}_{\text{int}} + \mathbf{r}_2)
  \big)
\Big].
$$

$$
\begin{aligned}
E_1(p_1, p_2; u, v)
&= \int_{\bm{\Omega}(u+p_1)}
   \overline{I}_u(\tau)\,\mathrm{d}\tau
 + \mu_1
   \int_{\bm{\Omega}(v+p_2)}
   \overline{I}_v(\tau)\,\mathrm{d}\tau \\
&\quad
 + \mu_2
   \int_{\bm{\Omega}(u - v + p_1 - p_2)}
   \overline{I}_d(\tau)\,\mathrm{d}\tau.
\end{aligned}
$$

This optimization problem is solved using a constrained
L-BFGS-B algorithm with box constraints.


In [ ]:
integrated = True
d = 2
R_int = 1.5
lam = 1.0

search_kwargs = dict(
    k=40,
    nms_size=40,
    exclude_center_radius=30.0,
    min_separation=10.0,
    refine_model='tps',
    refine_halfwin=3,
    tps_coarse_step=0.1,
    energy_halfwin=1.5,
    min_dist=20.0,
    antipodal_tol=10.0,
    angle_min_deg=20.0,
    w_exclude_center_radius=30.0,
    integrated=integrated, d=d, R_int=R_int, lam=lam
)

res = find_min_stable_patch_size_centered(
    binary_deformed,
    U_ref=u_shift,
    V_ref=v_shift,
    show_tracks=True,
    search_kwargs=search_kwargs,
    min_ps=20,
    max_ps=280,
    step=2,
    stable_seq_len=20,
    stable_tol=0.5,
)



## 1.5. Affinity calculation and adjustment

### 1.5.1 Calculates the 6 local affinities at the center

To rectify the affine transformation, we utilize the six points forming a hexagon
in the auto-correlation matrix. These points correspond to the shifts between
the three overlaid patterns in our image. Our rectification problem can be
formulated as a system of linear equations:

$$
A \cdot b = c
$$

where

$$
A =
\begin{pmatrix}
x_1 & y_1 & 0   & 0 \\
0   & 0   & x_1 & y_1 \\
x_2 & y_2 & 0   & 0 \\
0   & 0   & x_2 & y_2
\end{pmatrix},
\quad
b =
\begin{pmatrix}
M_{11} \\
M_{12} \\
M_{21} \\
M_{22}
\end{pmatrix},
\quad
c =
\begin{pmatrix}
x'_1 \\
y'_1 \\
x'_2 \\
y'_2
\end{pmatrix}.
$$


In [ ]:
local_affinities, patch_centers, min_patch_found = local_affinity_via_min_patch_at(
    image_deformee=binary_deformed,
    center_rc=[500,500],
    U_ref=u_shift,
    V_ref=v_shift,
    min_ps=20,
    max_ps=280,
    step=1,
    stable_seq_len=20,
    stable_tol=1,     
    hex_kwargs=search_kwargs,

    which_pair="all",
    verbose=False,
    show_tracks=False,
    strict_inside=True,
    pad_mode="reflect",
)

### 1.5.2. Choice of affinity based on phase correlation

Using phase correlation, we take each affinity among the six calculated affinities, then we correct the min_patch_found by each of them in order to select the corrected image most correlated with the reference image, which gives the correct affinity reflecting the correct match between the hexagon from the autocorrelation and the reference hexagon.

In [ ]:
redr_best, H_used_best = choose_best_affinity_patch(
    min_stable_patch=min_patch_found,
    affs=local_affinities,
    image_ref=binary_dots_texture
)

### 1.5.3. Alignment and visualization

In [ ]:
imm_transl, correlation,ref_resized, crop_resized = crop_and_align(binary_dots_texture,redr_best)

### Error between original image and corrected patch

In [ ]:
display_img(binary_dots_texture-imm_transl)

# 2. Homography Case

In this section, we will present the case of homography and how we can use autocorrelation and 6-peak deformation to correct a homographic deformation.

## 2.1. generate and deforme texture

### 2.1.1 white noise doubly-shifted texture

In [ ]:
shape = 1000
u_shift = (60, 0)
v_shift = (0, 60)

seed=21353456

noise_texture = generate_white_noise_and_shifts(
        shape, shape, u_shift, v_shift,seed=seed
    )
noise_texture = cv.GaussianBlur(noise_texture,(11,11),0)
display_img(noise_texture)

### 2.1.2 random binary dots doubly-shifted texture

In [ ]:
shape = 1000
dilation_size_i = 1
density = 0.01
angle_shift = 90
norm_shift = 60

binary_dots_texture = gen_random_binary_texture(shape,shape, dilation_size_i,density, angle_shift, norm_shift)
display_img(binary_dots_texture)

## 2.2. texture deformation

In [ ]:
scale = 1
rotation = 0
tilt = 1
tilt_orientation = 0
inclination_x = 10
inclination_y = 10


noise_deformed, noise_matrix = general_deformer(noise_texture,scale, rotation, tilt, tilt_orientation, inclination_x, inclination_y)
binary_deformed, binary_matrix = general_deformer(binary_dots_texture,scale, rotation, tilt, tilt_orientation, inclination_x, inclination_y)
display_img(noise_deformed)
display_img(binary_deformed)

## 2.3. autocorrelation

In [ ]:
noise_autocorrelation = autocorrelation_display(noise_texture)
deformed_noise_autocorrelation = autocorrelation_display(noise_deformed)
display_img(deformed_noise_autocorrelation)

In [ ]:
binary_autocorrelation = autocorrelation_display(binary_dots_texture)
binary_deformed_autocorrelation = autocorrelation_display(binary_deformed)
display_img(binary_deformed_autocorrelation)

## 2.4. Peaks detection and find min stable patch

In [ ]:
integrated = True
d = 2
R_int = 1.5
lam = 1.0

search_kwargs = dict(
    k=40,
    nms_size=40,
    exclude_center_radius=30.0,
    min_separation=10.0,
    refine_model='tps',
    refine_halfwin=3,
    tps_coarse_step=0.1,
    energy_halfwin=1.5,
    min_dist=20.0,
    antipodal_tol=10.0,
    angle_min_deg=20.0,
    w_exclude_center_radius=30.0,
    integrated=integrated, d=d, R_int=R_int, lam=lam
)

res = find_min_stable_patch_size_centered(
    binary_deformed,
    U_ref=u_shift,
    V_ref=v_shift,
    show_tracks=True,
    search_kwargs=search_kwargs,
    min_ps=100,
    max_ps=280,
    step=2,
    stable_seq_len=10,
    stable_tol=1,
)



In [ ]:
ps_all = sorted(res["ordered6_per_ps"].keys())
if len(ps_all) < 2:
    raise ValueError("Not enough points.")

M_all = [res["ordered6_per_ps"][p] for p in ps_all]

dM_all = [np.linalg.norm(M_all[i] - M_all[i-1]) for i in range(1, len(M_all))]
ps_dM  = ps_all[1:]

plt.figure(figsize=(10, 4))
plt.plot(ps_dM, dM_all, marker="o", label="||Mean hexagon Δ||")

if res.get("mid_ps", None) is not None:
    plt.axvline(res["mid_ps"], color="k", linestyle="--", label="mid_ps")

if res.get("min_stable_ps", None) is not None:
    min_stable_ps = res["min_stable_ps"]
    step = res["params"]["step"]
    k = res["params"]["stable_seq_len"]
    plt.axvspan(min_stable_ps, min_stable_ps + k * step, alpha=0.15, label="stable window")

plt.xlabel("Patch size (ps)")
plt.ylabel("Mean variation")
plt.title("stability of the mean hexagon (pair_mean)")
plt.ylim(0, 5)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
history = [h for h in res["history"] if h.get("ok", False)]

ps_all = [h["ps"] for h in history]
du_all = [h["du"] for h in history]
dv_all = [h["dv"] for h in history]
dw_all = [h["dw"] for h in history]

min_ps = res["min_stable_ps"]
step = res["params"]["step"]
k = res["params"]["stable_seq_len"]

max_ps = min_ps + (k - 1) * step

stable_idx = [
    i for i, p in enumerate(ps_all)
    if min_ps <= p <= max_ps
]

plt.figure(figsize=(10,4))
plt.plot(ps_all, du_all, color="blue", alpha=0.4, label="dv (global)")
plt.plot(ps_all, dv_all, color="orange", alpha=0.4, label="dv (global)")
plt.plot(ps_all, dw_all, color="green",  alpha=0.4, label="dw (global)")

plt.plot(
    [ps_all[i] for i in stable_idx],
    [du_all[i] for i in stable_idx],
    color="blue", linewidth=2.5, label="du (stable)"
)
plt.plot(
    [ps_all[i] for i in stable_idx],
    [dv_all[i] for i in stable_idx],
    color="orange", linewidth=2.5, label="dv (stable)"
)
plt.plot(
    [ps_all[i] for i in stable_idx],
    [dw_all[i] for i in stable_idx],
    color="green", linewidth=2.5, label="dw (stable)"
)

plt.axvline(res["mid_ps"], color="k", linestyle="--", label="mid_ps")

plt.xlabel("Patch size")
plt.ylabel("Δ vecteurs")
plt.ylim(0,20)
plt.title("dv / dw")
plt.legend()
plt.grid(True)
plt.show()


After extracting peaks from the hexagon and applying the minimal patch ensuring the stability of these peaks, calculating local affinities in a "center_rc" position (example below in [500,500]) gives us the six possible affinities for each position. We then need to use phase correlation to determine the affinity.

In [ ]:
local_affinities, patch_centers, min_patch_found = local_affinity_via_min_patch_at(
    image_deformee=binary_deformed,
    center_rc=[500,500],
    U_ref=u_shift,
    V_ref=v_shift,
    min_ps=20,
    max_ps=200,
    step=1,
    stable_seq_len=10,
    stable_tol=1,     
    hex_kwargs=search_kwargs,

    which_pair="all",
    verbose=False,
    show_tracks=False,
    strict_inside=True,
    pad_mode="reflect",
)


## 2.5. Patch rectification

In this section, we can see patch-by-patch rectification using the same approach: 

#### autocorrelation -> peak detection -> affinity calculation -> phase correlation and alignment -> rectification.

### 2.5.1 Calculation of local affinities

In [ ]:
affinities = []
stable_patchs = []

list_positions = oprt.generate_positions_in_roi(binary_deformed, 200, 800, 200, 800, mode="grid", step=200)

for i in list_positions :
    local_affinities, patch_centers, min_patch_found = local_affinity_via_min_patch_at(
        image_deformee=binary_deformed,
        center_rc=i,
        U_ref=u_shift,
        V_ref=v_shift,
        min_ps=20,
        max_ps=280,
        step=1,
        stable_seq_len=10,
        stable_tol=1,     
        hex_kwargs=search_kwargs,

        which_pair="all",
        verbose=False,
        show_tracks=False,
        strict_inside=True,
        pad_mode="reflect",
    )

    redr_best, H_used_best = choose_best_affinity_patch(
        min_stable_patch=min_patch_found,
        affs=local_affinities,
        image_ref=binary_dots_texture
    )
    affinities.append(H_used_best)
    stable_patchs.append(redr_best)

### 2.5.2 Phase correlation alignment

In [ ]:
imm_transl_list = []
for i in  stable_patchs :
    imm_transl, correlation,ref_resized, crop_resized = crop_and_align(binary_dots_texture,i)
    imm_transl_list.append(imm_transl)

### 2.5.3 error between the original image and the image corrected by patches

In [ ]:
image_f = imm_transl_list[0] 
for i in range(1,len(imm_transl_list)): 
    image_f += imm_transl_list[i] 
    image_f = image_f


In [ ]:
display_img(binary_dots_texture-image_f,width=600)

## 2.6. Rectification by homography estimation

### 2.6.1 Calculation of local affinities and estimation

**Estimation of the homography.**

After calculating local affinities, we seek an approximation of the overall deformation using these affinities retrieved with the positions of the corresponding patch centers by solving the following equation 

$$
\mathrm{H}^\ast
=
\underset{\mathrm{H}\in\mathcal{H}}{\operatorname*{argmin}}
\;
\frac{1}{n}
\sum_{i=1}^n
\left\|
\nabla_{y_i} \mathrm{H}
-
\mathbf{A}_i^{-1}
\right\|_F
$$


In [ ]:
final_image, total_h_estim = rectification(
    binary_dots_texture,
    binary_deformed,
    iteration=1,
    U_ref=(60, 0),
    V_ref=(0, 60),

    num_centers=40,
    border_margin=200,
    rng_seed=None,

    min_ps=60,
    max_ps=280,
    step=2,
    stable_seq_len=10,
    stable_tol=2,   
    use_cond_hard_filter=True,
    cond_max=2.0,
    use_cond_robust_filter=True,
    k_mad_cond=2.0,
    use_A_robust_filter=True,
    k_mad_A=2.0,
)


#### Log10(Error) frobenius between True and estimated homography

In [ ]:
print(np.log10(frobenius_norm(total_h_estim,np.linalg.inv(binary_matrix))))

In [ ]:
total_h_estim,np.linalg.inv(binary_matrix)

### 2.6.2 Phase correlation alignment

In [ ]:
image_transl, _,__, ___ = crop_and_align(binary_dots_texture,final_image)

### 2.6.3 Error between the original image and the corrected image

In [ ]:
display_img(binary_dots_texture-image_transl,width=600)

# 3. Study of affinity numbers and estimation

Let's take a grid of local affinities 

In [ ]:
affinities_and_centers_by_iter = []

list_positions = oprt.generate_positions_in_roi(binary_deformed, 200, 800, 200, 800, mode="grid", step=200)

for i in list_positions :
    local_affinities, patch_centers, min_patch_found = local_affinity_via_min_patch_at(
        image_deformee=binary_deformed,
        center_rc=i,
        U_ref=u_shift,
        V_ref=v_shift,
        min_ps=60,
        max_ps=280,
        step=1,
        stable_seq_len=10,
        stable_tol=2,     
        hex_kwargs=search_kwargs,

        which_pair="all",
        verbose=False,
        show_tracks=False,
        strict_inside=True,
        pad_mode="reflect",
    )
    redr_best, H_used_best = choose_best_affinity_patch(
        min_stable_patch=min_patch_found,
        affs=local_affinities,
        image_ref=binary_dots_texture
    )
    affinities_and_centers_by_iter.append((H_used_best, patch_centers[0],redr_best))


In [ ]:
arrays_only = [A for (A, _,_) in affinities_and_centers_by_iter]
points_only = [i for (_,i, _) in affinities_and_centers_by_iter]
im_only = [M for (_,_, M) in affinities_and_centers_by_iter]

phase correlation and alignment

In [ ]:
p_transl_list = []
for A,i,M in  zip(arrays_only,points_only,im_only) :
    p_transl, correlation,ref_resized, crop_resized = crop_and_align(binary_dots_texture,M)
    p_transl_list.append(p_transl)

image_p = p_transl_list[0] 
for i in range(1,len(p_transl_list)): 
    image_p += p_transl_list[i] 
    image_p = image_p


Heat map of the number of Jacobin conditioning values

In [ ]:
H = binary_matrix
out_shape = binary_deformed.shape
cond = cond_number_map_homography(H, out_shape, use_inverse=True)
plot_cond_heatmap(cond, img_warped=None, log_scale=True)
display_img(binary_deformed)

From the heat map of the number of Jacobin conditioning values for the homographic deformation applied, we can see that local affinities can have a minimum conditioning value of 1 and a maximum value that is not far from 1. We define conditioning filters on local affinities to verify their calculation.

In [ ]:
def filter_affinities_and_centers(
    local_affinities,
    patch_centers,
    *,
    stride: int = 1,
    max_samples: int | None = None,
    use_cond_hard_filter: bool = True,
    cond_max: float = 100.0,
    use_cond_robust_filter: bool = True,
    k_mad_cond: float = 3.0,
    use_A_robust_filter: bool = True,
    k_mad_A: float = 3.0,
    verbose: bool = True,
):
    n_aff = len(local_affinities)
    n_ctr = len(patch_centers)
    n = min(n_aff, n_ctr)

    if verbose:
        print("len(local_affinities):", n_aff, "| len(patch_centers):", n_ctr)

    if n < 3:
        if verbose:
            print("[filter] Not enough affinities/centers.")
        return [], [], {"reason": "not_enough_pairs", "n": n}

    if stride < 1:
        raise ValueError("stride must be >= 1")

    A_candidates, b_candidates, cond_candidates = [], [], []
    used = 0

    for idx in range(0, n, stride):
        aff = local_affinities[idx]
        r, c = patch_centers[idx]  # (row, col)

        M = np.asarray(aff, float)
        if M.ndim != 2 or M.shape[0] < 2 or M.shape[1] < 2:
            continue

        A = M[:2, :2]
        if not np.isfinite(A).all():
            continue

        try:
            condA = np.linalg.cond(A)
        except Exception:
            continue

        if use_cond_hard_filter and condA > cond_max:
            continue

        A_candidates.append(A)
        b_candidates.append(np.array([float(c), float(r)], dtype=float))  # (x,y)=(col,row)
        cond_candidates.append(float(condA))

        used += 1
        if max_samples is not None and used >= max_samples:
            break

    if len(A_candidates) < 3:
        if verbose:
            print(f"[filter] < 3 valid candidates after base filters: {len(A_candidates)}")
        return [], [], {"reason": "not_enough_after_base_filters", "n_candidates": len(A_candidates)}

    cond_candidates = np.array(cond_candidates, dtype=float)

    # Robust filter on cond(A) (NO log10)
    A_tmp, b_tmp = A_candidates, b_candidates
    if use_cond_robust_filter and len(cond_candidates) >= 3:
        med_c = np.median(cond_candidates)
        mad_c = np.median(np.abs(cond_candidates - med_c)) + 1e-9
        mask_cond = np.abs(cond_candidates - med_c) <= k_mad_cond * mad_c

        A_tmp = [A for A, m in zip(A_candidates, mask_cond) if m]
        b_tmp = [b for b, m in zip(b_candidates, mask_cond) if m]

        if verbose:
            print(f"[filter] After cond-robust (no log10): {len(A_tmp)} / {len(A_candidates)}")

        if len(A_tmp) < 3:
            if verbose:
                print("[filter] Too few after cond-robust -> fallback to candidates.")
            A_tmp, b_tmp = A_candidates, b_candidates

    # Robust filter on A coefficients
    A_filtered, b_filtered = A_tmp, b_tmp
    if use_A_robust_filter and len(A_tmp) >= 3:
        A_stack = np.array([A.flatten() for A in A_tmp], dtype=float)  # (N,4)
        med_A = np.median(A_stack, axis=0)
        mad_A_vec = np.median(np.abs(A_stack - med_A), axis=0) + 1e-9

        diff = np.abs(A_stack - med_A)
        mask_A = (diff <= k_mad_A * mad_A_vec).all(axis=1)

        A_filtered = [A for A, m in zip(A_tmp, mask_A) if m]
        b_filtered = [b for b, m in zip(b_tmp, mask_A) if m]

        if verbose:
            print(f"[filter] After A-robust: {len(A_filtered)} / {len(A_tmp)}")

        if len(A_filtered) < 3:
            if verbose:
                print("[filter] Too few after A-robust -> fallback to A_tmp.")
            A_filtered, b_filtered = A_tmp, b_tmp

    if len(A_filtered) < 3:
        if verbose:
            print("[filter] Still < 3 pairs, stopping.")
        return [], [], {"reason": "not_enough_after_filters", "n_filtered": len(A_filtered)}

    return A_filtered, b_filtered



def estimate_homography_from_Ab(
    A_list,
    b_list,
    *,
    optimize_fn,
    verbose: bool = True,
):

    if optimize_fn is None or not callable(optimize_fn):
        raise ValueError("optimize_fn must be a callable function: optimize_fn(A_list, b_list)")

    H_raw, cost = optimize_fn(A_list, b_list)

    H_estim = np.vstack((H_raw.T, np.array([[0, 0, 1]]))).T
    if abs(H_estim[2, 2]) > 1e-12:
        H_estim /= H_estim[2, 2]

    if verbose:
        try:
            print("[estimate_H] cost:", float(cost))
        except Exception:
            print("[estimate_H] cost:", cost)

    return H_estim


def apply_and_accumulate_homography(
    current_image,
    reference_image,
    H_estim,
    *,
    oprt,
    total_H=None,
    borderValue: int | None = None,
):
    if total_H is None:
        total_H = np.eye(3, dtype=float)

    if borderValue is None:
        borderValue = int(np.mean(current_image))

    new_total_H = H_estim @ total_H
    if abs(new_total_H[2, 2]) > 1e-12:
        new_total_H /= new_total_H[2, 2]

    new_image, H_used_candidate = oprt.centralizer(
        current_image,
        new_total_H,
        reference_image.shape,
        border_val=0,
    )

    return new_image, new_total_H


In [ ]:
A_filt, b_filt = filter_affinities_and_centers(
    arrays_only,
    points_only,
    stride=1,
    use_cond_hard_filter=False,
    cond_max=2.0,
    use_cond_robust_filter=False,
    k_mad_cond=2.0,
    use_A_robust_filter=False,
    k_mad_A=2.0,
    verbose=True
)


A_array = np.array(A_filt)   # (N,2,2)
N = len(A_array)
x = np.arange(N)

plt.figure(figsize=(10, 4))
plt.plot(x, A_array[:, 0, 0], marker="o", label="a11")
plt.plot(x, A_array[:, 0, 1], marker="o", label="a12")
plt.plot(x, A_array[:, 1, 0], marker="o", label="a21")
plt.plot(x, A_array[:, 1, 1], marker="o", label="a22")
plt.xlabel("Affinity index")
plt.ylabel("Coefficient value")
plt.title("Affinity Matrix Coefficients")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

cond_values = [np.linalg.cond(A) for A in A_filt]
plt.figure(figsize=(8, 4))
plt.scatter(x, cond_values)
plt.xlabel("Affinity index")
plt.ylabel("Condition number")
plt.title("Condition Number of Each Affinity Matrix")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
A_filt, b_filt = filter_affinities_and_centers(
    arrays_only,
    points_only,
    stride=1,
    use_cond_hard_filter=True,
    cond_max=2.0,
    use_cond_robust_filter=True,
    k_mad_cond=3.0,
    use_A_robust_filter=True,
    k_mad_A=3.0,
    verbose=True
)
A_filt, b_filt

#### Here, taking the calculated affinities, we estimate the homography by starting with 3 random affinities among those calculated, then we increase the number of affinities regularly. 
#### For a number N of affinities, we take n_trials times a number N of affinities at random and then continue the estimation.
#### The objective is to see the impact of the choice of affinity positions on the overall homography estimation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from joblib import Parallel, delayed

def _normalize_H(H):
    H = np.asarray(H, float)
    if abs(H[2, 2]) > 1e-12:
        H = H / H[2, 2]
    return H

def _apply_H_to_points(H, pts_xy):
    H = np.asarray(H, float)
    pts = np.c_[pts_xy, np.ones((len(pts_xy), 1), float)]
    q = (H @ pts.T).T
    q = q[:, :2] / q[:, 2:3]
    return q

def _estimate_H_from_subset(A_filt, b_filt, idxs, optimize_fn):
    A_sub = [A_filt[i] for i in idxs]
    b_sub = [b_filt[i] for i in idxs]
    H_raw, cost = optimize_fn(A_sub, b_sub)
    H = np.vstack((H_raw.T, np.array([[0.0, 0.0, 1.0]]))).T
    return _normalize_H(H), cost

def _one_trial(
    k: int,
    seed: int,
    *,
    N: int,
    A_filt,
    b_filt,
    optimize_fn,
    H_ref,
    corners,
    corners_ref,
):
    rng = np.random.default_rng(seed)
    idxs = rng.choice(N, size=k, replace=False)

    Hk, _ = _estimate_H_from_subset(A_filt, b_filt, idxs, optimize_fn)
    Hk[0, 2] = H_ref[0, 2]
    Hk[1, 2] = H_ref[1, 2]

    corners_k = _apply_H_to_points(Hk, corners)
    err = float(np.mean(np.linalg.norm(corners_k - corners_ref, axis=1)))
    return err, Hk

def _compute_for_k_parallel(
    k: int,
    *,
    N: int,
    A_filt,
    b_filt,
    optimize_fn,
    H_ref,
    corners,
    corners_ref,
    n_trials: int,
    base_seed: int,
    n_jobs_trials: int,
):
    seeds = [base_seed + 100000 * int(k) + t for t in range(int(n_trials))]

    trials = Parallel(n_jobs=n_jobs_trials, backend="loky", verbose=0)(
        delayed(_one_trial)(
            k, s,
            N=N,
            A_filt=A_filt,
            b_filt=b_filt,
            optimize_fn=optimize_fn,
            H_ref=H_ref,
            corners=corners,
            corners_ref=corners_ref,
        )
        for s in seeds
    )

    errs = np.array([t[0] for t in trials], dtype=float)
    best_i = int(np.argmin(errs))
    err_mean = float(errs.mean())
    err_min = float(errs[best_i])
    best_H = trials[best_i][1]
    return int(k), err_mean, err_min, best_H


N = len(A_filt)
if N < 3:
    raise ValueError("Il faut au moins 3 affinités/points dans A_filt/b_filt.")

H_ref = _normalize_H(np.linalg.inv(binary_matrix))

corners = np.array([[0, 0], [199, 0], [199, 199], [0, 199]], dtype=float)
corners_ref = _apply_H_to_points(H_ref, corners)

ks = np.arange(3, N + 1)
n_trials = 6
base_seed = 0

n_jobs_k = -1
n_jobs_trials = 1

results = Parallel(n_jobs=n_jobs_k, backend="loky", verbose=0)(
    delayed(_compute_for_k_parallel)(
        int(k),
        N=N,
        A_filt=A_filt,
        b_filt=b_filt,
        optimize_fn=optimize,
        H_ref=H_ref,
        corners=corners,
        corners_ref=corners_ref,
        n_trials=n_trials,
        base_seed=base_seed,
        n_jobs_trials=n_jobs_trials,
    )
    for k in ks
)

results.sort(key=lambda t: t[0])

errs_mean = np.array([r[1] for r in results], dtype=float)
errs_min  = np.array([r[2] for r in results], dtype=float)
best_H_for_k = {r[0]: r[3] for r in results}


To review unused settings and details

In [ ]:
plt.figure()
plt.plot(ks, errs_mean, label="Mean error (corners)")
plt.plot(ks, errs_min, label="Best error (corners)")
plt.xlabel("Number of affinities/points used (k)")
plt.ylabel("Mean error on the 4 corners (pixels)")
plt.title("Error between inv(H_true) and H_estim via corners (200x200)")
plt.grid(True)
plt.ylim(0, 20)
plt.legend()
plt.show()


show_ks = range(6)
show_ks = [k for k in show_ks if k in best_H_for_k]

plt.figure()

cmap = plt.cm.Blues
norm = plt.Normalize(vmin=min(show_ks), vmax=max(show_ks))

ref_poly = np.vstack([corners_ref, corners_ref[0]])
plt.plot(
    ref_poly[:, 0],
    ref_poly[:, 1],
    color="green",
    linewidth=2,
    label="Corners with inv(H_true)"
)

for k in show_ks:
    Hk = best_H_for_k[k].copy()
    Hk[0, 2] = H_ref[0, 2]
    Hk[1, 2] = H_ref[1, 2]

    ck = _apply_H_to_points(Hk, corners)
    poly = np.vstack([ck, ck[0]])

    plt.plot(
        poly[:, 0],
        poly[:, 1],
        color=cmap(norm(k)),
        linewidth=1.8,
        label=f"k = {k}"
    )

plt.xlabel("x (pixels)")
plt.ylabel("y (pixels)")
plt.title("Rectified corners (blue gradient according to k)")
plt.grid(True)
plt.gca().set_aspect("equal", adjustable="box")
plt.legend()
plt.show()
